# Shufflestorm Runner & Settings Configurator

This notebook clones the **Shufflestorm** project, allows you to configure its execution parameters in `settings.py`, runs both pipelines, and displays the results.

### Objectives:
1. **Clone** the Shufflestorm repository.
2. **Configure** all parameters in `shufflestorm/config/settings.py`.
3. **Execute Part 1** — the All Pairs matching pipeline (`allpairs.py`).
4. **Display** the All Pairs performance results.
5. **Configure Part 2** — set the relation size for the Triple Join.
6. **Execute Part 2** — the Triple Join benchmark (`triplejoin.py`).
7. **Display** the Triple Join performance results.

## 1. Clone the Repository

In [ ]:
import os
import shutil

repo_url = "https://github.com/Gatmatz/shufflestorm.git"
repo_dir = "shufflestorm_cloned"

# Clean up existing clone to ensure a fresh setup
if os.path.exists(repo_dir):
    print(f"Removing existing directory '{repo_dir}' to ensure fresh clone...")
    shutil.rmtree(repo_dir)

print(f"Cloning {repo_url} into '{repo_dir}'...")
!git clone {repo_url} {repo_dir}

# Change directory into the cloned repository
%cd {repo_dir}

## 2. Configure Settings (All Pairs)

Modify the fields below to customize the run. This cell will automatically update the `shufflestorm/config/settings.py` file with your parameters.

Available datasets in the repository:
- `bikedekho-small` (Size: 500)
- `bikedekho` (Size: 4786)
- `bikewale` (Size: 9003)
- `citeseer` (Size: 200000)

In [ ]:
# ==========================================
# Configure your settings here:
# ==========================================

DATASET = "bikewale"        # Dataset name (e.g. "bikedekho-small", "bikedekho", "bikewale", "citeseer")
DATASET_SIZE = 9003         # Number of records in the dataset (500, 4786, 9003, 200000)
REDUCER_SIZE = 200          # For Group-Based Matcher (number of records per reducer)
P_PRIME = 3                 # For Afrati-Ullman Model (prime number for mapping)

# ==========================================
# Write the configuration file
# ==========================================
settings_content = f"""# Dataset Parameters

DATASET = \"{DATASET}\"
DATASET_SIZE = {DATASET_SIZE}

# For Group-Based Matcher
REDUCER_SIZE = {REDUCER_SIZE}

# For Afrati-Ullman Model
P_PRIME = {P_PRIME}
"""

config_path = os.path.join("shufflestorm", "config", "settings.py")

with open(config_path, "w") as f:
    f.write(settings_content)

print(f"Successfully wrote new configuration to {config_path}:")
print("-" * 40)
print(settings_content)
print("-" * 40)

## 3. Install Dependencies (if needed)

If PySpark or other required packages are not installed in your environment, you can install them using this cell.

In [ ]:
import sys

# Check PySpark installation and install if missing
try:
    import pyspark
    print("PySpark is already installed.")
except ImportError:
    print("PySpark not found. Installing dependencies...")
    !{sys.executable} -m pip install pyspark numpy pandas python-dotenv

---

# Part 1 — All Pairs

## 4. Run the All Pairs Pipeline

Run `allpairs.py` using the current python executable. This will run:
1. **Naïve Strategy**
2. **Group-based Strategy**
3. **Spark SQL & Optimizer**
4. **Afrati-Ullmann Strategy**

The execution metrics (Execution Time, Task Count, Shuffle Volume) will be printed below and saved to the `results/` directory as JSON.

In [ ]:
import sys

print("Running allpairs.py ...")
!{sys.executable} allpairs.py

## 5. Visualize All Pairs Results

Read the generated JSON results file and display it in a comparison DataFrame.

In [ ]:
import json
import os
import pandas as pd

results_file = f"results/{DATASET}_results.json"

if os.path.exists(results_file):
    with open(results_file, "r") as f:
        data = json.load(f)
    
    print("Configuration Used:")
    for k, v in data["settings"].items():
        print(f"  {k}: {v}")
    print("\nStrategy Performance Comparison:")
    
    df_results = pd.DataFrame(data["results"]).T
    # Format the columns for readability
    if "time" in df_results.columns:
        df_results["time"] = df_results["time"].map(lambda x: f"{x:.4f} s" if isinstance(x, (int, float)) else x)
    if "shuffle_volume_bytes" in df_results.columns:
        df_results["shuffle_volume_bytes"] = df_results["shuffle_volume_bytes"].map(lambda x: f"{x:,} bytes" if isinstance(x, (int, float)) else x)
    
    display(df_results)
else:
    print(f"Results file '{results_file}' not found. Make sure allpairs.py ran successfully.")

---

# Part 2 — Triple Join

Computes the ternary join **A(x,y) ⋈ B(y,z) ⋈ C(z,w)** using three strategies:
1. **Direct ternary join** — a single-pass, three-way join.
2. **Two consecutive binary joins** — decomposed as (A ⋈ B) ⋈ C.
3. **SQL query** — executed via Spark SQL.

All relations are generated with the same configurable size.

## 6. Configure Triple Join

Set the number of rows for each relation (A, B, C). This cell overwrites the `RELATION_SIZE` variable inside `triplejoin.py`.

In [ ]:
# ==========================================
# Configure the relation size here:
# ==========================================

RELATION_SIZE = 1000  # Number of rows for each of A, B, C

# ==========================================
# Patch triplejoin.py with the chosen size
# ==========================================
import re

tj_path = "triplejoin.py"
with open(tj_path, "r") as f:
    content = f.read()

content = re.sub(
    r"RELATION_SIZE\s*=\s*\d+",
    f"RELATION_SIZE = {RELATION_SIZE}",
    content,
)

with open(tj_path, "w") as f:
    f.write(content)

print(f"RELATION_SIZE set to {RELATION_SIZE} in {tj_path}")

## 7. Run the Triple Join Pipeline

Run `triplejoin.py` using the current python executable. This will run:
1. **Direct Ternary Join**
2. **Two Consecutive Binary Joins**
3. **Spark SQL Join**

Metrics are printed below and saved to `results/<RELATION_SIZE>_triplejoin_results.json`.

In [ ]:
import sys

print("Running triplejoin.py ...")
!{sys.executable} triplejoin.py

## 8. Visualize Triple Join Results

Read the generated JSON results and display a comparison DataFrame.

In [ ]:
import json
import os
import pandas as pd

tj_results_file = f"results/{RELATION_SIZE}_triplejoin_results.json"

if os.path.exists(tj_results_file):
    with open(tj_results_file, "r") as f:
        tj_data = json.load(f)
    
    print("Configuration Used:")
    for k, v in tj_data["settings"].items():
        print(f"  {k}: {v}")
    print("\nTriple Join Strategy Comparison:")
    
    df_tj = pd.DataFrame(tj_data["results"]).T
    if "time" in df_tj.columns:
        df_tj["time"] = df_tj["time"].map(lambda x: f"{x:.4f} s" if isinstance(x, (int, float)) else x)
    if "shuffle_volume_bytes" in df_tj.columns:
        df_tj["shuffle_volume_bytes"] = df_tj["shuffle_volume_bytes"].map(lambda x: f"{x:,} bytes" if isinstance(x, (int, float)) else x)
    
    display(df_tj)
else:
    print(f"Results file '{tj_results_file}' not found. Make sure triplejoin.py ran successfully.")